In [ ]:
%load_ext autoreload
%autoreload 2

# CICE + MOM6 regional case setup

Sets up a real CrocoDash case with a CICE-coupled compset. CICE's forcing is
now a single expanded-grid restoring file (see `extract_forcings/cice.py`'s
`process_cice_forcing`): the case's domain plus a halo (`n_halo_cells` on
every side), built by ESMF-regridding a real global CICE restart onto that
expanded grid, with the restart snapshot held constant across the run's time
axis -- there's no separate CICE IC/OBC anymore. It's fully wired into
`Case.configure_forcings`/`process_forcings()` via `CICEConfigurator`, so it
runs automatically alongside MOM6's own forcings below -- no follow-up call
needed.

## Step 0: Paths and imports

Follows the "Fast CrocoDash Case Setup" pattern from the repo's `CLAUDE.md`.

In [ ]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.topo import Topo
from CrocoDash.vgrid import VGrid
from CrocoDash.case import Case

CESMROOT = Path("~/work/installs/full_regional_cesm").expanduser()
CASENAME = "cice_obc_demo"
CASEROOT = (Path("~/croc_cases") / CASENAME).expanduser()
INPUTDIR = (Path("~/scratch/croc_input") / CASENAME).expanduser()

print(CESMROOT)
print(CASEROOT)
print(INPUTDIR)

## Step 1: Grid, topography, vertical grid

A small Bering Sea domain -- high enough latitude that the reference CICE
restart used for CICE OBCs later actually has sea ice there in January, but
not so far north that it needs the polar (EPSG:3995) grid machinery. Flat
bathymetry + uniform vgrid, same as the CLAUDE.md fast-setup pattern -- this
is a wiring demo, not a science case.

In [ ]:
grid = Grid(
    lenx=15.0,
    leny=10.0,
    resolution=1.0,
    xstart=-170.0,
    ystart=68.0,
    name="cice_obc_demo",
)
topo = Topo(grid, min_depth=10.0, git=False)
topo.set_flat(1000.0)
vgrid = VGrid.uniform(nk=10, depth=1000.0, name="cice_obc_demo")

## Step 2: Create the case

Swap `SICE` for `CICE` in the NYF compset (per
`demos/gallery/notebooks/CrocoDash/features/coupling/add_cice.ipynb`) to get
a real coupled sea-ice model instead of the stub.

In [ ]:
case = Case(
    cesmroot=CESMROOT,
    caseroot=CASEROOT,
    inputdir=INPUTDIR,
    compset="1850_DATM%NYF_SLND_CICE_MOM6%REGIONAL_SROF_SGLC_SWAV",
    ocn_grid=grid,
    ocn_topo=topo,
    ocn_vgrid=vgrid,
    atm_grid_name="T62",
    machine="derecho",
    project="ncgd0011",
    override=True,
)
print("CICE in compset:", case.cice_in_compset)

## Step 3: Configure + process forcings

`configure_forcings` sets up the MOM6 IC/OBC download (GLORYS via RDA on
this system, no extra credentials needed) plus CICE's `restart_path`/
`grid_path`/`n_halo_cells` -- `CICEConfigurator`'s own inputs, required since
CICE is in the compset. `process_forcings()` then runs the full driver
workflow for everything active for this compset, including CICE's new
`process_cice_forcing`.

In [ ]:
# Same reference tx2_3v3 global CICE restart + grid used in
# tests/extract_forcings/test_processes.py's test_process_cice_forcing_produces_output.
CICE_RESTART_PATH = (
    "/glade/u/home/dbailey/"
    "b.e30_alpha09b.B1850C_MTso.ne30_t233_wgx3.360.cice.r.0201-01-01-00000.nc"
)
CICE_GRID_PATH = "/glade/campaign/cesm/community/omwg/grids/tx2_3v3_grid.nc"

case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    restart_path=CICE_RESTART_PATH,
    grid_path=CICE_GRID_PATH,
    n_halo_cells=2,
)

In [ ]:
case.process_forcings()

## Case is set up

MOM6's own IC/OBC and CICE's `cice_forcing.nc` (covering the domain plus its
2-cell halo) are both in `case.inputdir / "ocnice"` now -- both generated by
the single `process_forcings()` call above.

In [ ]:
print("caseroot:", case.caseroot)
print("inputdir:", case.inputdir)
print("supergrid_path:", case.supergrid_path)

## Inspect CICE's forcing file

Confirms the output covers the domain plus its halo, spans the case's date
range, and has real (nonzero) sea ice -- this Bering Sea domain has real
January sea ice cover in the reference restart.

In [ ]:
import xarray as xr

cice_ds = xr.open_dataset(case.inputdir / "ocnice" / "cice_forcing.nc")
print("dims:", dict(cice_ds.sizes))
print("domain (T-cells):", grid.tlon.shape, "-> with halo:", (cice_ds.sizes["ny"], cice_ds.sizes["nx"]))
print("time:", cice_ds["time"].values)
print("total sea ice area (aicen sum):", float(cice_ds["aicen"].sum()))